# <div style="text-align:center; color:#76B900;">AIQToolkit NVIDIA Blueprint</div>

<div style="text-align:center;">
    <img src="https://raw.githubusercontent.com/NVIDIA/cuda-python/main/docs/resources/nvidia_logo.png" width="200" alt="NVIDIA Logo"/>
</div>

<div style="text-align:center; margin-top:20px; margin-bottom:20px;">
    <span style="background-color:#76B900; color:white; padding:5px 10px; border-radius:4px; margin-right:5px;">NVIDIA Blueprint</span>
    <span style="background-color:#333; color:white; padding:5px 10px; border-radius:4px; margin-right:5px;">AI Monitoring</span>
    <span style="background-color:#333; color:white; padding:5px 10px; border-radius:4px;">Control Tower</span>
</div>

Welcome to the AIQToolkit NVIDIA Blueprint - your interactive control center for monitoring and interacting with NVIDIA AI services.

This notebook will guide you through setting up and launching the NVIDIA Blueprint with an elegant, intuitive interface. Follow along to unlock the full potential of your NVIDIA-powered AI workflow.

> **Note**: This blueprint runs best on NVIDIA GPU-accelerated machines but will gracefully adapt to non-GPU environments.

## 🔍 Environment Analysis

Let's begin by examining your environment and detecting available resources. This helps us optimize the blueprint for your specific setup.

In [ ]:
# Import necessary libraries
import os
import sys
import json
import time
import platform
import subprocess
import importlib.util
import IPython
from IPython.display import display, HTML, clear_output, Markdown
import threading
from pathlib import Path

# Check Python version
python_version = platform.python_version()
sufficient_python = tuple(map(int, python_version.split('.'))) >= (3, 8)

# Display the environment info with custom styling
display(HTML(f"""
<div style="background-color:#f8f9fa; padding:15px; border-radius:5px; margin:10px 0;">
    <h3 style="color:#76B900; margin-top:0;">🖥️ System Environment</h3>
    <table style="width:100%; border-collapse:collapse;">
        <tr>
            <td style="padding:8px; border-bottom:1px solid #ddd; width:30%;"><strong>Operating System</strong></td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{platform.system()} {platform.release()}</td>
        </tr>
        <tr>
            <td style="padding:8px; border-bottom:1px solid #ddd;"><strong>Python Version</strong></td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{python_version} {'✅' if sufficient_python else '❌'}</td>
        </tr>
        <tr>
            <td style="padding:8px; border-bottom:1px solid #ddd;"><strong>Hostname</strong></td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{platform.node()}</td>
        </tr>
        <tr>
            <td style="padding:8px; border-bottom:1px solid #ddd;"><strong>Architecture</strong></td>
            <td style="padding:8px; border-bottom:1px solid #ddd;">{platform.machine()}</td>
        </tr>
        <tr>
            <td style="padding:8px;"><strong>Processor</strong></td>
            <td style="padding:8px;">{platform.processor() or 'Unknown'}</td>
        </tr>
    </table>
</div>
"""))

# Add a helper class for visualization
class VisualOutput:
    @staticmethod
    def success(message):
        display(HTML(f'<div style="background-color:#e8f5e9; padding:10px; border-left:4px solid #4caf50; border-radius:4px; margin:10px 0;"><span style="color:#2e7d32;">✅ {message}</span></div>'))
        
    @staticmethod
    def warning(message):
        display(HTML(f'<div style="background-color:#fff8e1; padding:10px; border-left:4px solid #ffc107; border-radius:4px; margin:10px 0;"><span style="color:#ff8f00;">⚠️ {message}</span></div>'))
        
    @staticmethod
    def error(message):
        display(HTML(f'<div style="background-color:#ffebee; padding:10px; border-left:4px solid #f44336; border-radius:4px; margin:10px 0;"><span style="color:#c62828;">❌ {message}</span></div>'))
    
    @staticmethod
    def info(message):
        display(HTML(f'<div style="background-color:#e3f2fd; padding:10px; border-left:4px solid #2196f3; border-radius:4px; margin:10px 0;"><span style="color:#0d47a1;">{message}</span></div>'))
    
    @staticmethod
    def code(message):
        display(HTML(f'<div style="background-color:#f5f5f5; padding:10px; border-radius:4px; font-family:monospace; margin:10px 0;">{message}</div>'))

## 🔧 Installing Dependencies

Next, we'll ensure all necessary dependencies are installed. This might take a moment, but it's a one-time setup.

In [ ]:
# Install required packages
!pip install -q requests flask pillow ipywidgets matplotlib psutil nvidia-ml-py3

try:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    import requests
    import psutil
    VisualOutput.success("All dependencies installed successfully!")
except ImportError as e:
    VisualOutput.error(f"Failed to import dependency: {str(e)}")

## 🔍 NVIDIA GPU Detection

Let's check if your environment has NVIDIA GPUs available. The blueprint works best with GPU acceleration, but will adapt to CPU-only environments.

In [ ]:
def check_nvidia_gpu():
    """Comprehensive NVIDIA GPU detection with rich output"""
    try:
        import pynvml
        pynvml.nvmlInit()
        
        gpu_count = pynvml.nvmlDeviceGetCount()
        if gpu_count == 0:
            raise Exception("No NVIDIA GPUs detected with PyNVML")
        
        gpu_info = []
        total_memory = 0
        total_compute = 0
        
        for i in range(gpu_count):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            name = pynvml.nvmlDeviceGetName(handle)
            memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
            compute_capability = pynvml.nvmlDeviceGetCudaComputeCapability(handle)
            
            total_memory += memory.total
            memory_gb = memory.total / (1024**3)
            
            temperature = pynvml.nvmlDeviceGetTemperature(handle, 0)
            utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)
            
            gpu_info.append({
                "index": i,
                "name": name.decode() if isinstance(name, bytes) else name,
                "memory_total_gb": memory_gb,
                "compute_capability": f"{compute_capability[0]}.{compute_capability[1]}",
                "temperature": temperature,
                "utilization": utilization.gpu
            })
            
            total_compute += compute_capability[0] * 10 + compute_capability[1]
        
        # Create GPU info display
        html_output = f"""
        <div style="background-color:#f0fff0; padding:15px; border-radius:8px; margin:10px 0; border:1px solid #76B900;">
            <h3 style="color:#76B900; margin-top:0;">🚀 NVIDIA GPU Detected</h3>
            <p>Found <strong>{gpu_count}</strong> NVIDIA GPU{'s' if gpu_count > 1 else ''} with a total of <strong>{total_memory / (1024**3):.2f} GB</strong> memory</p>
            
            <table style="width:100%; border-collapse:collapse; margin-top:10px;">
                <tr style="background-color:#76B900; color:white;">
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">GPU</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">Model</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">Memory</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">Compute</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">Temp</th>
                    <th style="padding:8px; text-align:left; border:1px solid #ddd;">Usage</th>
                </tr>
        """
        
        for idx, gpu in enumerate(gpu_info):
            html_output += f"""
                <tr style="background-color:{'#f9f9f9' if idx % 2 == 0 else 'white'};">
                    <td style="padding:8px; border:1px solid #ddd;">{gpu['index']}</td>
                    <td style="padding:8px; border:1px solid #ddd;">{gpu['name']}</td>
                    <td style="padding:8px; border:1px solid #ddd;">{gpu['memory_total_gb']:.2f} GB</td>
                    <td style="padding:8px; border:1px solid #ddd;">CUDA {gpu['compute_capability']}</td>
                    <td style="padding:8px; border:1px solid #ddd;">{gpu['temperature']}°C</td>
                    <td style="padding:8px; border:1px solid #ddd;">
                        <div style="background-color:#e0e0e0; width:100%; height:20px; border-radius:4px;">
                            <div style="background-color:#76B900; width:{gpu['utilization']}%; height:20px; border-radius:4px;"></div>
                        </div>
                        {gpu['utilization']}%
                    </td>
                </tr>
            """
            
        html_output += """
            </table>
            <div style="margin-top:15px;">
                <strong style="color:#76B900;">✓ GPU Acceleration:</strong> Available and ready for AI workloads
            </div>
        </div>
        """
        
        display(HTML(html_output))
        return True, gpu_info
    
    except Exception as e:
        try:
            # Fall back to nvidia-smi if PyNVML fails
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            if result.returncode == 0:
                VisualOutput.success("NVIDIA GPU detected through nvidia-smi")
                output = result.stdout.strip()
                VisualOutput.code(output.replace('\n', '<br>').replace(' ', '&nbsp;'))
                return True, {"output": output}
        except Exception:
            pass
            
        VisualOutput.warning(f"No NVIDIA GPU detected or drivers not properly installed. The blueprint will run in CPU-only mode.")
        display(HTML(f"""
        <div style="background-color:#fff8e1; padding:15px; border-radius:8px; margin:10px 0; border:1px solid #ffca28;">
            <h3 style="color:#f57c00; margin-top:0;">⚠️ No NVIDIA GPU Detected</h3>
            <p>The blueprint will run in CPU-only mode with limited performance.</p>
            <p><strong>Reason:</strong> {str(e)}</p>
            <div style="margin-top:15px;">
                <strong>For optimal experience:</strong>
                <ul>
                    <li>Ensure NVIDIA drivers are installed</li>
                    <li>Check if CUDA toolkit is available</li>
                    <li>Verify the GPU is properly connected</li>
                </ul>
            </div>
        </div>
        """))
        return False, None

# Run GPU detection with visual output
has_gpu, gpu_info = check_nvidia_gpu()

## 🔑 NVIDIA API Key Configuration

To interact with NVIDIA services, you can provide an API key. This is optional but enables additional functionality.

In [ ]:
# Create a styled configuration interface with ipywidgets
style = {'description_width': 'initial'}
layout = widgets.Layout(width='100%')

# Header
header = widgets.HTML(
    value="""
    <div style="padding:10px; margin-bottom:10px; border-radius:5px; background-color:#f8f8f8;">
        <h3 style="margin-top:0; color:#76B900;">NVIDIA Configuration</h3>
        <p>Configure your NVIDIA API credentials for enhanced functionality</p>
    </div>
    """
)

# Endpoint URL input with default
endpoint_input = widgets.Text(
    value='https://jupyter0-s1ondnjfx.brevlab.com',
    placeholder='Enter NVIDIA endpoint URL',
    description='NVIDIA Endpoint:',
    style=style,
    layout=layout
)

# API Key input (password field for security)
api_key_input = widgets.Password(
    value='',
    placeholder='Enter your NVIDIA API key (optional)',
    description='NVIDIA API Key:',
    style=style,
    layout=layout
)

# Save configuration checkbox
save_checkbox = widgets.Checkbox(
    value=True,
    description='Save configuration for this session',
    style=style
)

# Advanced options accordion
advanced_options = widgets.Accordion([
    widgets.VBox([
        widgets.IntSlider(
            value=8080,
            min=1024,
            max=65535,
            description='UI Port:',
            style=style,
            layout=layout
        ),
        widgets.IntSlider(
            value=8000,
            min=1024,
            max=65535,
            description='API Port:',
            style=style,
            layout=layout
        ),
        widgets.Checkbox(
            value=True,
            description='Enable GPU acceleration (if available)',
            style=style
        ),
        widgets.Checkbox(
            value=True,
            description='Auto-open browser',
            style=style
        )
    ])
])
advanced_options.set_title(0, 'Advanced Options')

# Save button with styling
save_button = widgets.Button(
    description='Save Configuration',
    button_style='success',
    tooltip='Save configuration and continue',
    icon='check'
)

# Output area for messages
output = widgets.Output()

# Handle save button click
def on_save_button_clicked(b):
    with output:
        clear_output()
        
        # Validate endpoint URL
        endpoint = endpoint_input.value.strip()
        if not endpoint.startswith('http'):
            VisualOutput.error("Endpoint URL must start with http:// or https://")
            return
            
        # Validate API key format (if provided)
        api_key = api_key_input.value.strip()
        if api_key and not (api_key.startswith('nvapi-') or len(api_key) > 20):
            VisualOutput.warning("The provided API key doesn't match the expected NVIDIA API key format")
        
        # Save configuration
        os.environ['NVIDIA_ENDPOINT'] = endpoint
        if api_key:
            os.environ['NVIDIA_API_KEY'] = api_key
            VisualOutput.success(f"NVIDIA API Key configured: {api_key[:5]}...")
        else:
            VisualOutput.info("No NVIDIA API Key provided. Some features may be limited.")
            
        VisualOutput.success(f"NVIDIA Endpoint configured: {endpoint}")
        
        if save_checkbox.value:
            # Save to .env file in the current directory
            with open('.env', 'w') as f:
                f.write(f"NVIDIA_ENDPOINT={endpoint}\n")
                if api_key:
                    f.write(f"NVIDIA_API_KEY={api_key}\n")
            VisualOutput.info("Configuration saved to .env file")
            
        time.sleep(1)  # Brief pause for better UX
        
        # Display success message with animation
        display(HTML("""
        <div style="text-align:center; margin:20px;">
            <div style="font-size:24px; color:#76B900;">
                <span style="display:inline-block; animation: pulse 1.5s infinite;">✓</span> Configuration Saved!
            </div>
            <style>
                @keyframes pulse {
                    0% { transform: scale(1); }
                    50% { transform: scale(1.2); }
                    100% { transform: scale(1); }
                }
            </style>
        </div>
        <div style="text-align:center;">
            <p>Ready to launch the blueprint. Continue to the next step!</p>
        </div>
        """))

# Connect button click handler
save_button.on_click(on_save_button_clicked)

# Display the configuration interface
display(header)
display(endpoint_input)
display(api_key_input)
display(save_checkbox)
display(advanced_options)
display(save_button)
display(output)

## 🚀 Launch the Blueprint

Now let's launch the AIQToolkit NVIDIA Blueprint with our configuration. This will start the web servers and provide interactive access to the Control Tower.

In [ ]:
# Create dynamic card with server information
server_info = widgets.Output()

# Global variables for server processes and state
processes = []
api_server_process = None
ui_server_process = None
api_port = 8000
ui_port = 8080
api_running = False
ui_running = False

# Function to launch servers
def launch_servers(start_button=None):
    global api_server_process, ui_server_process, api_running, ui_running, processes
    
    with server_info:
        clear_output()
        display(HTML("""
        <div style="padding:15px; border-radius:5px; background-color:#f5f5f5; margin-bottom:15px;">
            <h3 style="margin-top:0; color:#333;">🚀 Launching AIQToolkit NVIDIA Blueprint</h3>
            <div id="launch-progress" style="margin:15px 0;">
                <div style="height:6px; background-color:#e0e0e0; border-radius:3px; overflow:hidden;">
                    <div id="progress-bar" style="width:10%; height:100%; background-color:#76B900; animation: progress-animation 10s ease-in-out;"></div>
                </div>
                <style>
                    @keyframes progress-animation {
                        0% { width: 0%; }
                        50% { width: 50%; }
                        100% { width: 100%; }
                    }
                </style>
                <p id="status-message" style="text-align:center; color:#666; margin:5px 0;">Initializing servers...</p>
            </div>
        </div>
        """))
        
        if start_button:
            start_button.disabled = True
        
        # Find the web-ui directory (try current directory or parent directories)
        web_ui_dir = None
        potential_paths = [
            Path.cwd(),
            Path.cwd() / "web-ui",
            Path.cwd().parent / "web-ui",
            Path.cwd().parent.parent / "web-ui"
        ]
        
        for path in potential_paths:
            if (path / "public").exists() and (path / "api").exists():
                web_ui_dir = path
                break
                
        if not web_ui_dir:
            VisualOutput.error("Could not find web-ui directory with public and api subdirectories")
            if start_button:
                start_button.disabled = False
            return False, False
        
        VisualOutput.info(f"Found web-ui directory at: {web_ui_dir}")
        
        # Create logs directory if needed
        logs_dir = web_ui_dir / "logs"
        logs_dir.mkdir(exist_ok=True)
        
        # Set up environment variables
        env = os.environ.copy()
        if 'NVIDIA_API_KEY' in os.environ:
            env['NVIDIA_API_KEY'] = os.environ['NVIDIA_API_KEY']
        if 'NVIDIA_ENDPOINT' in os.environ:
            env['NVIDIA_ENDPOINT'] = os.environ['NVIDIA_ENDPOINT']
            
        # Launch API server
        try:
            api_log_file = open(logs_dir / "api_server.log", "w")
            api_cmd = [
                sys.executable, "-m", "http.server", 
                str(api_port), 
                "--directory", str(web_ui_dir / "api")
            ]
            
            api_server_process = subprocess.Popen(
                api_cmd,
                stdout=api_log_file,
                stderr=subprocess.STDOUT,
                env=env,
                cwd=str(web_ui_dir)
            )
            processes.append(api_server_process)
            time.sleep(1)  # Brief pause to let server start
            
            if api_server_process.poll() is None:  # None means process is still running
                api_running = True
                VisualOutput.success(f"API server started on port {api_port}")
            else:
                VisualOutput.error(f"API server failed to start")
        except Exception as e:
            VisualOutput.error(f"Error starting API server: {str(e)}")
            
        # Launch Web UI server
        try:
            ui_log_file = open(logs_dir / "ui_server.log", "w")
            ui_cmd = [
                sys.executable, "-m", "http.server", 
                str(ui_port), 
                "--directory", str(web_ui_dir / "public")
            ]
            
            ui_server_process = subprocess.Popen(
                ui_cmd,
                stdout=ui_log_file,
                stderr=subprocess.STDOUT,
                env=env,
                cwd=str(web_ui_dir)
            )
            processes.append(ui_server_process)
            time.sleep(1)  # Brief pause to let server start
            
            if ui_server_process.poll() is None:  # None means process is still running
                ui_running = True
                VisualOutput.success(f"UI server started on port {ui_port}")
            else:
                VisualOutput.error(f"UI server failed to start")
        except Exception as e:
            VisualOutput.error(f"Error starting UI server: {str(e)}")
        
        # Show final status
        status_html = """
        <div style="margin-top:20px; padding:15px; border-radius:5px; background-color:#f0f0f0;">
            <h3 style="margin-top:0; color:#333;">Blueprint Status</h3>
            <table style="width:100%; border-collapse:collapse;">
        """
        
        status_html += f"""
                <tr>
                    <td style="padding:8px; border-bottom:1px solid #ddd; width:30%;"><strong>API Server</strong></td>
                    <td style="padding:8px; border-bottom:1px solid #ddd;">
                        <span style="color:{'green' if api_running else 'red'};">
                            {'✅ Running' if api_running else '❌ Not Running'}
                        </span>
                        {f'<a href="http://localhost:{api_port}/health" target="_blank" style="margin-left:10px;">Check Health</a>' if api_running else ''}
                    </td>
                </tr>
                <tr>
                    <td style="padding:8px; border-bottom:1px solid #ddd;"><strong>UI Server</strong></td>
                    <td style="padding:8px; border-bottom:1px solid #ddd;">
                        <span style="color:{'green' if ui_running else 'red'};">
                            {'✅ Running' if ui_running else '❌ Not Running'}
                        </span>
                    </td>
                </tr>
                <tr>
                    <td style="padding:8px; border-bottom:1px solid #ddd;"><strong>NVIDIA API Key</strong></td>
                    <td style="padding:8px; border-bottom:1px solid #ddd;">
                        <span style="color:{'green' if 'NVIDIA_API_KEY' in os.environ else 'orange'};">
                            {'✅ Configured' if 'NVIDIA_API_KEY' in os.environ else '⚠️ Not Configured'}
                        </span>
                    </td>
                </tr>
                <tr>
                    <td style="padding:8px;"><strong>GPU Acceleration</strong></td>
                    <td style="padding:8px;">
                        <span style="color:{'green' if has_gpu else 'orange'};">
                            {'✅ Available' if has_gpu else '⚠️ Not Available'}
                        </span>
                    </td>
                </tr>
            </table>
        </div>
        """
        
        display(HTML(status_html))
        
        # Enable button if it was disabled
        if start_button:
            start_button.disabled = False
            
    return api_running, ui_running

# Create styled launch button
launch_button = widgets.Button(
    description='Launch Blueprint',
    button_style='success',
    icon='rocket',
    layout=widgets.Layout(width='200px', height='40px')
)

# Connect button click handler
launch_button.on_click(lambda b: launch_servers(b))

# Display launch button and info area
display(widgets.VBox([
    widgets.HTML(value="<div style='margin:10px 0;'><p><strong>Click to launch the AIQToolkit NVIDIA Blueprint:</strong></p></div>"),
    launch_button
]))
display(server_info)

## 🖥️ Access the Control Tower UI

Once the servers are running, you can access the NVIDIA Control Tower UI to monitor your services. Click the button below to open the interface in a new tab.

In [ ]:
def create_access_buttons():
    """Create styled access buttons for the different interfaces"""
    if not ui_running:
        VisualOutput.warning("UI server is not running. Please launch the blueprint first.")
        return
    
    control_tower_url = f"http://localhost:{ui_port}/control-tower.html"
    console_url = f"http://localhost:{ui_port}/blueprint-console.html"
    
    buttons_html = f"""
    <div style="display:flex; flex-wrap:wrap; gap:20px; justify-content:center; margin-top:20px;">
        <a href="{control_tower_url}" target="_blank" style="text-decoration:none;">
            <div style="width:250px; height:180px; background-color:#ffffff; border-radius:10px; box-shadow:0 4px 6px rgba(0,0,0,0.1); overflow:hidden; transition:all 0.3s; border:1px solid #ddd;">
                <div style="height:40px; background-color:#76B900; color:white; display:flex; align-items:center; justify-content:center; font-weight:bold;">
                    NVIDIA Control Tower
                </div>
                <div style="padding:15px; text-align:center;">
                    <div style="font-size:40px; margin:10px 0;">🖥️</div>
                    <p style="margin:10px 0; color:#333;">Monitor NVIDIA services performance and status</p>
                </div>
            </div>
        </a>
        
        <a href="{console_url}" target="_blank" style="text-decoration:none;">
            <div style="width:250px; height:180px; background-color:#ffffff; border-radius:10px; box-shadow:0 4px 6px rgba(0,0,0,0.1); overflow:hidden; transition:all 0.3s; border:1px solid #ddd;">
                <div style="height:40px; background-color:#333; color:white; display:flex; align-items:center; justify-content:center; font-weight:bold;">
                    Blueprint Console
                </div>
                <div style="padding:15px; text-align:center;">
                    <div style="font-size:40px; margin:10px 0;">⌨️</div>
                    <p style="margin:10px 0; color:#333;">Command-line style interface for blueprint control</p>
                </div>
            </div>
        </a>
        
        <a href="{control_tower_url.replace('control-tower.html', 'index.html')}" target="_blank" style="text-decoration:none;">
            <div style="width:250px; height:180px; background-color:#ffffff; border-radius:10px; box-shadow:0 4px 6px rgba(0,0,0,0.1); overflow:hidden; transition:all 0.3s; border:1px solid #ddd;">
                <div style="height:40px; background-color:#1976d2; color:white; display:flex; align-items:center; justify-content:center; font-weight:bold;">
                    Report Generator
                </div>
                <div style="padding:15px; text-align:center;">
                    <div style="font-size:40px; margin:10px 0;">📊</div>
                    <p style="margin:10px 0; color:#333;">Generate reports using NVIDIA AI models</p>
                </div>
            </div>
        </a>
    </div>
    <style>
        a:hover > div {
            transform: translateY(-5px);
            box-shadow: 0 6px 12px rgba(0,0,0,0.15);
        }
    </style>
    """
    
    display(HTML(buttons_html))

# Create a button to open interfaces
open_ui_button = widgets.Button(
    description='Show Access Options',
    button_style='info',
    icon='external-link',
    layout=widgets.Layout(width='200px')
)

access_output = widgets.Output()

def on_open_ui_button_clicked(b):
    with access_output:
        clear_output()
        create_access_buttons()

open_ui_button.on_click(on_open_ui_button_clicked)

display(widgets.VBox([
    widgets.HTML(value="<div style='margin:10px 0;'><p><strong>Access the NVIDIA Blueprint interfaces:</strong></p></div>"),
    open_ui_button
]))
display(access_output)

## 📊 System Monitoring

Let's set up real-time system monitoring to track resource usage while the blueprint is running.

In [ ]:
# Create real-time system monitoring
monitoring_output = widgets.Output()

# Initialize monitoring state
monitoring_active = False
stop_monitoring = threading.Event()

def monitor_system():
    """Monitor system resources and update the display"""
    global monitoring_active
    
    if monitoring_active:
        return
    
    monitoring_active = True
    stop_monitoring.clear()
    
    # Set up the plot
    plt.style.use('ggplot')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Data storage
    times = []
    cpu_percentages = []
    memory_percentages = []
    
    # Maximum data points to show
    max_points = 60
    
    # GPU data if available
    gpu_utilizations = []
    gpu_memories = []
    has_gpu_monitor = False
    
    try:
        import pynvml
        pynvml.nvmlInit()
        gpu_count = pynvml.nvmlDeviceGetCount()
        if gpu_count > 0:
            has_gpu_monitor = True
            handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # Just monitor the first GPU for simplicity
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))
            ax3.set_title('GPU Utilization')
            ax3.set_ylabel('Utilization %')
            ax3.set_ylim(0, 100)
    except:
        has_gpu_monitor = False
    
    # Set up axes
    ax1.set_title('CPU Utilization')
    ax1.set_ylabel('Utilization %')
    ax1.set_ylim(0, 100)
    
    ax2.set_title('Memory Utilization')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Utilization %')
    ax2.set_ylim(0, 100)
    
    # Create line plots
    cpu_line, = ax1.plot(times, cpu_percentages, 'b-', label='CPU')
    mem_line, = ax2.plot(times, memory_percentages, 'g-', label='Memory')
    
    if has_gpu_monitor:
        gpu_util_line, = ax3.plot(times, gpu_utilizations, 'r-', label='GPU')
        gpu_mem_line, = ax3.plot(times, gpu_memories, 'orange', label='GPU Memory')
        ax3.legend()
    
    ax1.legend()
    ax2.legend()
    plt.tight_layout()
    
    start_time = time.time()
    
    # Monitoring loop
    def update_plot():
        nonlocal times, cpu_percentages, memory_percentages, gpu_utilizations, gpu_memories
        
        while not stop_monitoring.is_set():
            # Get current resource usage
            current_time = time.time() - start_time
            cpu_percent = psutil.cpu_percent()
            memory_percent = psutil.virtual_memory().percent
            
            # Append data
            times.append(current_time)
            cpu_percentages.append(cpu_percent)
            memory_percentages.append(memory_percent)
            
            # Get GPU info if available
            if has_gpu_monitor:
                try:
                    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
                    gpu_utilizations.append(util.gpu)
                    gpu_memories.append(mem_info.used / mem_info.total * 100)
                except:
                    gpu_utilizations.append(0)
                    gpu_memories.append(0)
            
            # Limit data points
            if len(times) > max_points:
                times = times[-max_points:]
                cpu_percentages = cpu_percentages[-max_points:]
                memory_percentages = memory_percentages[-max_points:]
                if has_gpu_monitor:
                    gpu_utilizations = gpu_utilizations[-max_points:]
                    gpu_memories = gpu_memories[-max_points:]
            
            # Update plot data
            cpu_line.set_data(times, cpu_percentages)
            mem_line.set_data(times, memory_percentages)
            
            if has_gpu_monitor:
                gpu_util_line.set_data(times, gpu_utilizations)
                gpu_mem_line.set_data(times, gpu_memories)
            
            # Adjust x-axis limits
            for ax in [ax1, ax2] + ([ax3] if has_gpu_monitor else []):
                ax.set_xlim(max(0, current_time - 60), current_time + 1)
                ax.figure.canvas.draw()
            
            time.sleep(1)
    
    # Start monitoring thread
    threading.Thread(target=update_plot, daemon=True).start()
    
    return fig

# Create monitoring controls
monitoring_button = widgets.Button(
    description='Start Monitoring',
    button_style='info',
    icon='chart-line'
)

def on_monitoring_button_clicked(b):
    global monitoring_active
    
    if monitoring_active:
        # Stop monitoring
        stop_monitoring.set()
        monitoring_active = False
        b.description = 'Start Monitoring'
        b.icon = 'chart-line'
        with monitoring_output:
            clear_output()
            display(HTML("<div style='padding:15px; text-align:center;'>System monitoring stopped</div>"))
    else:
        # Start monitoring
        with monitoring_output:
            clear_output()
            fig = monitor_system()
            plt.show()
        b.description = 'Stop Monitoring'
        b.icon = 'stop'

monitoring_button.on_click(on_monitoring_button_clicked)

# Display monitoring controls
display(widgets.VBox([
    widgets.HTML(value="<div style='margin:10px 0;'><p><strong>System Resource Monitoring:</strong></p></div>"),
    monitoring_button
]))
display(monitoring_output)

## 🔄 Test NVIDIA Services

Let's test the connection to NVIDIA services. This will verify that your API key and endpoint are working correctly.

In [ ]:
# Create a dashboard for NVIDIA service testing
test_output = widgets.Output()

def test_nvidia_services():
    """Test NVIDIA services with rich visualizations"""
    with test_output:
        clear_output()
        
        if not api_running:
            VisualOutput.error("API server is not running. Please launch the blueprint first.")
            return
            
        VisualOutput.info("<strong>Testing NVIDIA services...</strong>")
        
        # First check API health
        try:
            response = requests.get(f"http://localhost:{api_port}/health", timeout=5)
            if response.status_code == 200:
                health_data = response.json()
                VisualOutput.success(f"API health check successful: {health_data.get('status', 'unknown')}")
            else:
                VisualOutput.error(f"API health check failed with status code {response.status_code}")
                return
        except Exception as e:
            VisualOutput.error(f"API health check error: {str(e)}")
            return
        
        # Test each NVIDIA service
        services = [
            {"id": "nim", "name": "NVIDIA NIM", "description": "NVIDIA Inference Microservices"},
            {"id": "api", "name": "NVIDIA API", "description": "NVIDIA Foundation Models API"},
            {"id": "nemo", "name": "NVIDIA NeMo", "description": "NVIDIA NeMo Framework"},
            {"id": "triton", "name": "NVIDIA Triton", "description": "NVIDIA Triton Inference Server"}
        ]
        
        results = {}
        latencies = {}
        
        # Create progress display
        progress_html = HTML("""
        <div style="margin:20px 0;">
            <div style="background-color:#f5f5f5; border-radius:5px; padding:15px;">
                <h3 style="margin-top:0; color:#333;">Testing Services</h3>
                <div id="test-progress"></div>
            </div>
        </div>
        """)
        display(progress_html)
        
        for idx, service in enumerate(services):
            service_id = service["id"]
            service_name = service["name"]
            
            # Update progress
            progress_percent = (idx / len(services)) * 100
            display(HTML(f"""
            <script>
                document.getElementById('test-progress').innerHTML = `
                    <p>Testing {service_name}...</p>
                    <div style="height:6px; background-color:#e0e0e0; border-radius:3px; overflow:hidden;">
                        <div style="width:{progress_percent}%; height:100%; background-color:#76B900;"></div>
                    </div>
                    <p style="text-align:center; font-size:12px; color:#666;">{idx+1} of {len(services)}</p>
                `;
            </script>
            """))
            
            try:
                # Make the test request
                start_time = time.time()
                response = requests.post(
                    f"http://localhost:{api_port}/nvidia",
                    json={
                        "prompt": "Hello, this is a test message to check connectivity.",
                        "service_type": service_id,
                        "model": "llama-3.1-70b-instruct",
                        "nvidia_api_key": os.environ.get("NVIDIA_API_KEY", ""),
                        "nvidia_endpoint": os.environ.get("NVIDIA_ENDPOINT", "")
                    },
                    timeout=10
                )
                latency = time.time() - start_time
                latencies[service_id] = latency
                
                if response.status_code == 200:
                    data = response.json()
                    status = data.get('status', 'unknown')
                    
                    results[service_id] = {
                        "success": status == 'success',
                        "status": status,
                        "latency": latency,
                        "message": data.get('response', '')[:100] + '...' if len(data.get('response', '')) > 100 else data.get('response', '')
                    }
                else:
                    results[service_id] = {
                        "success": False,
                        "status": "http_error",
                        "latency": latency,
                        "message": f"HTTP Error {response.status_code}"
                    }
            except Exception as e:
                results[service_id] = {
                    "success": False,
                    "status": "error",
                    "latency": 0,
                    "message": str(e)
                }
        
        # Complete progress bar
        display(HTML("""
        <script>
            document.getElementById('test-progress').innerHTML = `
                <p>Testing complete</p>
                <div style="height:6px; background-color:#e0e0e0; border-radius:3px; overflow:hidden;">
                    <div style="width:100%; height:100%; background-color:#76B900;"></div>
                </div>
                <p style="text-align:center; font-size:12px; color:#666;">{len(services)} of {len(services)}</p>
            `;
        </script>
        """))
        
        # Display test results in a nice dashboard
        results_html = """
        <div style="margin:20px 0;">
            <div style="background-color:#f8f8f8; border-radius:5px; padding:15px;">
                <h3 style="margin-top:0; color:#333;">NVIDIA Services Test Results</h3>
                <div style="display:flex; flex-wrap:wrap; gap:15px; margin-top:15px;">
        """
        
        for service in services:
            service_id = service["id"]
            result = results.get(service_id, {"success": False, "status": "unknown", "latency": 0, "message": "Test not run"})
            
            background_color = "#e8f5e9" if result["success"] else "#ffebee"
            border_color = "#4caf50" if result["success"] else "#f44336"
            status_icon = "✅" if result["success"] else "❌"
            status_text = "Success" if result["success"] else "Failed"
            
            results_html += f"""
                <div style="flex:1; min-width:250px; background-color:{background_color}; border-radius:5px; border-left:4px solid {border_color}; padding:15px;">
                    <div style="display:flex; justify-content:space-between; align-items:center;">
                        <strong style="font-size:16px;">{service['name']}</strong>
                        <span>{status_icon} {status_text}</span>
                    </div>
                    <div style="color:#666; font-size:12px; margin:5px 0;">{service['description']}</div>
                    <div style="margin-top:10px;">
                        <div><strong>Latency:</strong> {result['latency']:.2f}s</div>
                        <div style="margin-top:5px;">
                            <strong>Message:</strong> <span style="font-size:12px;">{result['message']}</span>
                        </div>
                    </div>
                </div>
            """
            
        results_html += """
                </div>
            </div>
        </div>
        """
        
        display(HTML(results_html))
        
        # Add summary section
        success_count = sum(1 for r in results.values() if r["success"])
        
        summary_html = f"""
        <div style="margin:20px 0;">
            <div style="background-color:{'#e8f5e9' if success_count >= 2 else '#fff8e1' if success_count >= 1 else '#ffebee'}; 
                        border-radius:5px; padding:15px; text-align:center;">
                <h3 style="margin-top:0; color:{'#2e7d32' if success_count >= 2 else '#f57c00' if success_count >= 1 else '#c62828'};">Test Summary</h3>
                <div style="font-size:18px; margin:10px 0;">
                    {success_count} of {len(services)} services tested successfully
                </div>
                <div>
                    {'All systems operational!' if success_count == len(services) else
                     'Most services are working. Check failed services above.' if success_count >= 2 else
                     'Several services are not responding. Check configuration.' if success_count >= 1 else
                     'Service connectivity issues detected. Verify your API key and endpoint configuration.'}
                </div>
            </div>
        </div>
        """
        
        display(HTML(summary_html))

# Create test button
test_button = widgets.Button(
    description='Test NVIDIA Services',
    button_style='info',
    icon='vial',
    layout=widgets.Layout(width='200px')
)

# Connect button click handler
test_button.on_click(lambda b: test_nvidia_services())

# Display test button and output
display(widgets.VBox([
    widgets.HTML(value="<div style='margin:10px 0;'><p><strong>Test connection to NVIDIA services:</strong></p></div>"),
    test_button
]))
display(test_output)

## 🧹 Cleanup Resources

When you're done working with the blueprint, you can clean up all resources by running the cell below.

In [ ]:
def cleanup_resources():
    """Clean up all resources and stop servers"""
    global api_server_process, ui_server_process, processes, api_running, ui_running, monitoring_active, stop_monitoring
    
    # Stop monitoring if active
    if monitoring_active:
        stop_monitoring.set()
        monitoring_active = False
    
    # Stop all processes
    print("Stopping all servers...")
    
    for process in processes:
        try:
            if process and process.poll() is None:  # Process exists and is running
                process.terminate()
                process.wait(timeout=3)
        except:
            try:
                process.kill()  # Force kill if terminate doesn't work
            except:
                pass
    
    # Reset state
    processes = []
    api_server_process = None
    ui_server_process = None
    api_running = False
    ui_running = False
    
    # Display confirmation
    VisualOutput.success("All resources have been released. You can safely close this notebook.")

# Create cleanup button
cleanup_button = widgets.Button(
    description='Cleanup Resources',
    button_style='danger',
    icon='trash',
    layout=widgets.Layout(width='200px')
)

# Connect button click handler
cleanup_button.on_click(lambda b: cleanup_resources())

# Register cleanup to run when kernel is shut down
import atexit
atexit.register(cleanup_resources)

# Display cleanup button
display(widgets.VBox([
    widgets.HTML(value="<div style='margin:10px 0;'><p><strong>Cleanup all resources when finished:</strong></p></div>"),
    cleanup_button
]))

## 📖 Additional Resources

Thank you for exploring the AIQToolkit NVIDIA Blueprint! Here are additional resources to help you get the most out of your experience:

In [ ]:
# Display resources in an attractive format
display(HTML("""
<div style="background-color:#f8f8f8; padding:20px; border-radius:8px;">
    <h3 style="margin-top:0; color:#76B900;">Additional Resources</h3>
    
    <div style="display:flex; flex-wrap:wrap; gap:20px; margin-top:15px;">
        <div style="flex:1; min-width:250px; background-color:white; padding:15px; border-radius:5px; box-shadow:0 2px 5px rgba(0,0,0,0.1);">
            <h4 style="color:#333; margin-top:0; border-bottom:2px solid #76B900; padding-bottom:8px;">Documentation</h4>
            <ul style="padding-left:20px;">
                <li><a href="https://docs.aiqtoolkit.ai" target="_blank">AIQToolkit Documentation</a></li>
                <li><a href="https://developer.nvidia.com" target="_blank">NVIDIA Developer Resources</a></li>
                <li><a href="https://github.com/plturrell/aiqtoolkit-nvidia" target="_blank">GitHub Repository</a></li>
            </ul>
        </div>
        
        <div style="flex:1; min-width:250px; background-color:white; padding:15px; border-radius:5px; box-shadow:0 2px 5px rgba(0,0,0,0.1);">
            <h4 style="color:#333; margin-top:0; border-bottom:2px solid #76B900; padding-bottom:8px;">Tutorials</h4>
            <ul style="padding-left:20px;">
                <li><a href="#" onclick="alert('Coming soon!'); return false;">Custom Model Integration</a></li>
                <li><a href="#" onclick="alert('Coming soon!'); return false;">Advanced Control Tower Features</a></li>
                <li><a href="#" onclick="alert('Coming soon!'); return false;">Deploying to Production</a></li>
            </ul>
        </div>
        
        <div style="flex:1; min-width:250px; background-color:white; padding:15px; border-radius:5px; box-shadow:0 2px 5px rgba(0,0,0,0.1);">
            <h4 style="color:#333; margin-top:0; border-bottom:2px solid #76B900; padding-bottom:8px;">Community</h4>
            <ul style="padding-left:20px;">
                <li><a href="#" onclick="alert('Coming soon!'); return false;">Discord Server</a></li>
                <li><a href="https://github.com/plturrell/aiqtoolkit-nvidia/issues" target="_blank">Issue Tracker</a></li>
                <li><a href="mailto:support@aiqtoolkit.ai">Email Support</a></li>
            </ul>
        </div>
    </div>
    
    <div style="margin-top:20px; text-align:center; padding:15px; background-color:#76B900; border-radius:5px; color:white;">
        <strong>Thank you for trying the AIQToolkit NVIDIA Blueprint!</strong><br>
        We hope you enjoy exploring the capabilities of this powerful toolkit.
    </div>
</div>
"""))